# 05 Run AIFS-single 1.0 yourself on a T4 (Tier 2, experimental)

An attempt to run the open checkpoint `ecmwf/aifs-single-1.0` once with `anemoi-inference` on a free Kaggle or Colab T4 GPU, and to compare the result with the operational AIFS fields already fetched from open data.

## What this notebook does and does not show

Does: check the GPU, install `anemoi-inference`, try to install Flash Attention, write a run configuration, run one 72 h forecast inside a try block that records the error and the elapsed time, and, if a file came out, compare 2 m temperature against the open-data AIFS field for the same initialisation and lead over the Norwegian bbox.

Does not: verify anything against observations, and does not promise to work. There is no confirmed report of this checkpoint running on a T4. A clean failure with the reason recorded is an acceptable outcome of this notebook.

**Stop rule.** If Flash Attention or GPU memory fails and the run has not finished after 45 minutes, write the failure and the elapsed time into the last cell and stop. Do not spend the session on it.

## Facts to keep in mind

* The checkpoint is about 1.19 GB and is pulled from Hugging Face on first use (network).
* A T4 is a Turing GPU with 16 GB. It has no bfloat16 support in hardware, so precision must be `16` (float16), not `bf16`.
* `ANEMOI_INFERENCE_NUM_CHUNKS` set to 2 to 4 splits the attention computation to reduce peak memory, at the cost of time.
* Flash Attention wheels are built for Ampere and newer; on Turing the build may fail or fall back. The model card notes the fallback path.
* CPU inference is 10 to 100 times slower and is not attempted here.
* The model runs on the N320 reduced Gaussian grid. The open-data files are on a regular 0.25 degree grid, so the comparison needs a regrid step.

In [ ]:
!nvidia-smi

The next two cells use the network (PyPI). The second one tries to build Flash Attention; it may take long and may fail on a T4, which is reported rather than treated as fatal.

In [ ]:
# Network.
%pip install -q anemoi-inference ecmwf-opendata

In [ ]:
# Network, optional. Flash Attention targets Ampere and newer; a T4 is Turing.
import subprocess
import sys
import time

t_start = time.time()
try:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"],
                       capture_output=True, text=True, timeout=1800)
    if r.returncode == 0:
        print("flash-attn installed")
    else:
        print("flash-attn did not install (expected on a T4); anemoi-inference falls back to standard attention")
        print(r.stderr[-1500:])
except Exception as e:
    print("flash-attn install failed:", type(e).__name__, e)
print(f"elapsed {(time.time() - t_start) / 60:.1f} min")

In [ ]:
import os

os.environ["ANEMOI_INFERENCE_NUM_CHUNKS"] = "4"
print("ANEMOI_INFERENCE_NUM_CHUNKS =", os.environ["ANEMOI_INFERENCE_NUM_CHUNKS"])

## Choose the initialisation

The comparison only works for an initialisation that is also present as an open-data field file from notebook 01. The cell picks the newest AIFS-single field file if one exists, else falls back to a fixed date. Open data for the initial conditions is only available for about four days, so the date has to be recent.

In [ ]:
from pathlib import Path

import pandas as pd

CONFIG = Path(os.environ.get("AFV_CONFIG", "configs/default.yaml"))
try:
    from afv.config import load_config
    cfg = load_config(CONFIG)
    fields_dir = cfg.get_path("fields_dir")
    bbox = cfg["stations"]["bbox"]
except Exception:
    cfg = None
    fields_dir = Path("data/cache/fields")
    bbox = {"lon_min": 4.5, "lon_max": 20.0, "lat_min": 60.0, "lat_max": 70.5}

aifs_fields = sorted(fields_dir.glob("aifs-single_*.nc"))
if aifs_fields:
    import xarray as xr
    OPEN_DATA_FIELD = aifs_fields[-1]
    with xr.open_dataset(OPEN_DATA_FIELD) as ds:
        INIT = pd.Timestamp(ds.attrs["init_time"]).tz_localize(None)
else:
    OPEN_DATA_FIELD = None
    INIT = pd.Timestamp("2026-09-06T00:00:00")  # must be within the open-data retention window
LEAD_H = 72
OWN_RUN = Path("/kaggle/working/aifs_own_run.nc") if os.path.exists("/kaggle") else Path("aifs_own_run.nc")
print("init:", INIT, "| lead:", LEAD_H, "h | open-data field:", OPEN_DATA_FIELD, "| own output:", OWN_RUN)

## Run configuration

Written in the shape the model card's `run_AIFS_v1.ipynb` documents: checkpoint from Hugging Face, initial conditions from ECMWF open data, float16 on the GPU, NetCDF out. The `input` block is the part most likely to have moved between `anemoi-inference` releases; verify it against the current model card before blaming the GPU.

In [ ]:
# Verify against the current model card (huggingface.co/ecmwf/aifs-single-1.0, run_AIFS_v1.ipynb)
# and the anemoi-inference docs before changing anything else.
RUN_YAML = f"""
checkpoint:
  huggingface: ecmwf/aifs-single-1.0

date: {INIT.strftime('%Y-%m-%dT%H:%M:%S')}
lead_time: {LEAD_H}

device: cuda
precision: 16          # T4 is Turing: float16, not bf16

input:
  opendata:            # initial conditions from ECMWF open data, as in run_AIFS_v1.ipynb
    source: ecmwf

output:
  netcdf: {OWN_RUN}

env:
  ANEMOI_INFERENCE_NUM_CHUNKS: 4
"""
CFG_PATH = OWN_RUN.with_name("aifs_t4_run.yaml")
CFG_PATH.write_text(RUN_YAML.strip() + "\n", encoding="utf-8")
print(CFG_PATH)
print(RUN_YAML)

## Run

The next cell uses the network: the 1.19 GB checkpoint from Hugging Face and the initial conditions from data.ecmwf.int. Whatever happens, the error type, message and elapsed time are printed, which is the record the stop rule asks for.

In [ ]:
import time
import traceback

t_start = time.time()
RUN_OK = False
try:
    from anemoi.inference.config.run import RunConfiguration
    from anemoi.inference.runners.default import DefaultRunner
    config = RunConfiguration.load(str(CFG_PATH))
    runner = DefaultRunner(config)
    runner.execute()
    RUN_OK = OWN_RUN.exists()
    print(f"finished, output exists: {RUN_OK}, elapsed {(time.time() - t_start) / 60:.1f} min")
except Exception as e:
    print(f"FAILED after {(time.time() - t_start) / 60:.1f} min: {type(e).__name__}: {e}")
    traceback.print_exc(limit=3)
    print("Record this in the failure note below and stop if the 45 minute budget is used up.")

## Compare with the operational AIFS open-data field

Guarded: it only runs when both the own output and the open-data field for the same initialisation exist. The own output is on the N320 grid (one `values` dimension with `latitude` and `longitude` coordinates), so it is mapped to the 0.25 degree points in the bbox by nearest neighbour before differencing. Expect small non-zero differences even on success: different attention kernel, float16 versus the operational precision, and the nearest-neighbour regrid.

In [ ]:
import numpy as np

if OWN_RUN.exists() and OPEN_DATA_FIELD is not None and Path(OPEN_DATA_FIELD).exists():
    import xarray as xr
    from scipy.spatial import cKDTree

    LEAD_CMP = 24
    ref = xr.open_dataset(OPEN_DATA_FIELD)
    ref_t = ref["t2m"].sel(lead_h=LEAD_CMP)
    ref_t = ref_t.sel(lat=slice(bbox["lat_min"], bbox["lat_max"]), lon=slice(bbox["lon_min"], bbox["lon_max"]))

    own = xr.open_dataset(OWN_RUN)
    name = "2t" if "2t" in own else ("t2m" if "t2m" in own else None)
    if name is None:
        print("own output has no 2t/t2m variable; variables are", list(own.data_vars))
    else:
        da = own[name]
        tdim = "time" if "time" in da.dims else da.dims[0]
        target = np.datetime64(INIT + pd.Timedelta(hours=LEAD_CMP))
        own_t = da.sel({tdim: target}, method="nearest") if tdim == "time" else da.isel({tdim: LEAD_CMP // 6})
        if {"lat", "lon"} <= set(own_t.dims):
            own_on_ref = own_t.interp(lat=ref_t["lat"], lon=ref_t["lon"], method="nearest").values
        else:
            lat_c = own_t["latitude"].values if "latitude" in own_t.coords else own_t["lat"].values
            lon_c = own_t["longitude"].values if "longitude" in own_t.coords else own_t["lon"].values
            lon_c = np.where(lon_c > 180, lon_c - 360.0, lon_c)
            tree = cKDTree(np.column_stack([lat_c.ravel(), lon_c.ravel()]))
            glat, glon = np.meshgrid(ref_t["lat"].values, ref_t["lon"].values, indexing="ij")
            _, idx = tree.query(np.column_stack([glat.ravel(), glon.ravel()]))
            own_on_ref = own_t.values.ravel()[idx].reshape(glat.shape)
        diff = own_on_ref - ref_t.values
        print(f"2t at lead {LEAD_CMP} h over the bbox, own run minus open data:")
        print(f"  max abs difference {np.nanmax(np.abs(diff)):.3f} K")
        print(f"  RMSE              {np.sqrt(np.nanmean(diff ** 2)):.3f} K")
        print(f"  mean difference   {np.nanmean(diff):.3f} K over {diff.size} grid points")
else:
    print("comparison skipped: own run exists =", OWN_RUN.exists(), "| open-data field =", OPEN_DATA_FIELD)

## Failure note

Fill in after the run, success or not, so the attempt is on record:

* date of attempt, environment (Kaggle or Colab), GPU from `nvidia-smi`
* `anemoi-inference` version, whether `flash-attn` installed
* outcome: finished in N minutes, or failed after N minutes with the error type
* peak memory if visible, and the `ANEMOI_INFERENCE_NUM_CHUNKS` value used